# Tugas 2 — Manipulasi Data dengan Pandas

Pengolahan data transaksi POS menggunakan dataset yang sama dengan tugas sebelumnya. File ini berdiri sendiri dan membaca dataset dari CSV asli.

## Persiapan Data

Pandas digunakan untuk membaca dan mengolah dataset transaksi.

In [ ]:
import pandas as pd

data = pd.read_csv("paket_a_transaksi_pos_2024_2025.csv")

print("Ukuran data:", data.shape)
data.head()

## 1. Pembersihan dan Pemilihan Data

### 1.1 Membersihkan Harga

Titik dihapus sebagai pemisah ribuan, lalu koma diubah menjadi titik desimal. Setelah itu, tipe data diubah menjadi `float`.

In [ ]:
data["harga_satuan"] = (
    data["harga_satuan"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

data[["harga_satuan"]].head()

### 1.2 Mengubah Kuantitas Negatif

Boolean Indexing digunakan untuk memilih nilai negatif. Nilai tersebut diubah menjadi positif dengan `.loc[]` dan `.abs()`.

In [ ]:
jumlah_negatif = data["jumlah"] < 0

data.loc[jumlah_negatif, "jumlah"] = data.loc[jumlah_negatif, "jumlah"].abs()

print("Jumlah negatif tersisa:", (data["jumlah"] < 0).sum())

### 1.3 Mengisi Kode Kasir Kosong

Nilai kosong pada kolom `kode_kasir` diisi dengan teks `K_UNKNOWN`.

In [ ]:
kasir_kosong = data["kode_kasir"].isna()
data.loc[kasir_kosong, "kode_kasir"] = "K_UNKNOWN"

print("Kode kasir kosong:", data["kode_kasir"].isna().sum())

### 1.4 Memilih Data dengan `.iloc[]`

Menampilkan kolom `id_transaksi`, `tanggal`, `kode_gerai`, dan `jumlah` dari lima baris pertama.

In [ ]:
data.iloc[:5, [0, 1, 2, 5]]

## 2. Operasi Aritmatika

Total kotor dihitung dari harga dikali jumlah. Nominal diskon dihitung dari persentase diskon. Total bersih merupakan nilai setelah diskon ditambah PPN 11%.

In [ ]:
data["Total_Kotor"] = data["harga_satuan"] * data["jumlah"]
data["Nominal_Diskon"] = data["Total_Kotor"] * data["diskon_persen"] / 100
data["Total_Bersih"] = (data["Total_Kotor"] - data["Nominal_Diskon"]) * 1.11

data[["id_transaksi", "Total_Kotor", "Nominal_Diskon", "Total_Bersih"]].head()

## 3. Penyaringan Data

### 3.1 Transaksi Selesai dari Gerai G05 atau G09

Data dipilih jika statusnya `SELESAI`, kode gerainya G05 atau G09, dan total bersih lebih dari Rp100.000.

In [ ]:
filter_gerai = data["kode_gerai"].isin(["G05", "G09"])
filter_status = data["status"] == "SELESAI"
filter_omzet = data["Total_Bersih"] > 100_000

transaksi_terpilih = data.loc[filter_status & filter_gerai & filter_omzet]

print("Jumlah baris:", len(transaksi_terpilih))
transaksi_terpilih.head()

### 3.2 Transaksi Gerai G10

`.query()` digunakan untuk memilih transaksi gerai G10 dengan jumlah pembelian minimal lima unit.

In [ ]:
transaksi_g10 = data.query("kode_gerai == 'G10' and jumlah >= 5")

print("Jumlah baris:", len(transaksi_g10))
transaksi_g10.head()

## 4. Pengurutan Data

### 4.1 Mengurutkan Dua Kolom

Data diurutkan berdasarkan kode gerai A–Z, lalu total bersih dari terbesar ke terkecil.

In [ ]:
data_urut = data.sort_values(
    by=["kode_gerai", "Total_Bersih"],
    ascending=[True, False]
)

data_urut[["id_transaksi", "kode_gerai", "Total_Bersih"]].head()

### 4.2 Lima Transaksi Terbesar

`.nlargest()` digunakan untuk mengambil lima transaksi dengan total bersih tertinggi.

In [ ]:
data.nlargest(5, "Total_Bersih")[
    ["id_transaksi", "kode_gerai", "jumlah", "Total_Bersih"]
]

## 5. Pengelompokan dan Ringkasan Data

### 5.1 Laporan Omzet per Gerai

Laporan hanya memakai transaksi dengan status `SELESAI`. Setiap gerai dihitung jumlah transaksinya, total unit, total omzet bersih, dan rata-rata nilai transaksi.

In [ ]:
data_selesai = data.loc[data["status"] == "SELESAI"]

laporan_gerai = (
    data_selesai.groupby("kode_gerai")
    .agg(
        Jumlah_Transaksi=("id_transaksi", "count"),
        Total_Unit_Terjual=("jumlah", "sum"),
        Total_Omzet_Bersih=("Total_Bersih", "sum"),
        Rata_Rata_Transaksi=("Total_Bersih", "mean")
    )
    .sort_values("Total_Omzet_Bersih", ascending=False)
    .round(2)
)

laporan_gerai

### 5.2 Kontribusi Omzet Setiap Gerai

`.transform()` menghitung total omzet bersih per gerai. Kontribusi setiap transaksi dihitung dengan membagi total bersih transaksi dengan total bersih gerainya, lalu dikali 100.

In [ ]:
total_gerai = data.groupby("kode_gerai")["Total_Bersih"].transform("sum")

data["Kontribusi_Gerai_Persen"] = data["Total_Bersih"] / total_gerai * 100

data[[
    "id_transaksi",
    "kode_gerai",
    "Total_Bersih",
    "Kontribusi_Gerai_Persen"
]].head()

## Pemeriksaan Akhir

Pemeriksaan tipe harga, nilai kuantitas negatif, dan kode kasir kosong.

In [ ]:
print("Tipe harga:", data["harga_satuan"].dtype)
print("Kuantitas negatif:", (data["jumlah"] < 0).sum())
print("Kode kasir kosong:", data["kode_kasir"].isna().sum())
print("Jumlah baris:", len(data))